In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -q
!pip install transformers accelerate -q

In [2]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA GeForce RTX 3090
VRAM: 25.3 GB


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM


draft_model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(draft_model_name)
draft_model = AutoModelForCausalLM.from_pretrained(
    draft_model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
)

print(f"Loaded: {draft_model_name}")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

/opt/jupyterlab-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 316.24it/s]


Loaded: Qwen/Qwen3-1.7B
VRAM: 3.4 GB


## Test draft model

In [4]:
promt = "The capital of Russia is"
inputs = tokenizer(promt, return_tensors='pt').to("cuda")

with torch.no_grad():
    outputs = draft_model(**inputs)

logits = outputs.logits
logits.shape

torch.Size([1, 5, 151936])

In [5]:
last_logits = logits[0, -1, :]
last_logits

tensor([10.7578, 13.0547,  6.2461,  ...,  0.6685,  0.6685,  0.6685],
       device='cuda:0', dtype=torch.float16)

In [6]:
preds = torch.softmax(last_logits, dim = -1)
preds

tensor([1.0371e-05, 1.0329e-04, 1.1921e-07,  ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00], device='cuda:0', dtype=torch.float16)

In [7]:
top5 = torch.topk(preds, 5)

top5

torch.return_types.topk(
values=tensor([0.3862, 0.0515, 0.0401, 0.0370, 0.0359], device='cuda:0',
       dtype=torch.float16),
indices=tensor([22415,   279,  1304,  1112,   264], device='cuda:0'))

In [8]:
best_token_id = top5.indices[0].item()
best_prob = top5.values[0].item()
best_token_id

22415

In [9]:
tokenizer.decode(best_token_id)

' Moscow'

## Create generate func for draft model

In [129]:
def generate_recs_draft_model(model, text_before: str = '', n_tokens : int = 5):

    lst_probs = []
    lst_ids = []
    
    total_text = ''
    
    for _ in range(n_tokens):
    
        promt = text_before
        inputs = tokenizer(promt, return_tensors = 'pt').to('cuda')
    
        with torch.no_grad():
            outputs = model(**inputs)
    
        logits = outputs.logits
        last_logit = logits[0, -1, :]
    
        preds = torch.softmax(last_logit, dim = -1)
    
        best_token = torch.topk(preds, 1)
    
        best_indx = best_token.indices[0].item()
        best_prob = best_token.values[0].item()

        str_token = tokenizer.decode(best_indx)
        
        text_before += str_token
        total_text += str_token
        lst_ids.append(best_indx)
        lst_probs.append(best_prob)
    
    return text_before, total_text, lst_probs, lst_ids

In [130]:
generate_recs_draft_model(draft_model, 'The capital of Russia is')

('The capital of Russia is Moscow. The capital of',
 ' Moscow. The capital of',
 [0.38623046875, 0.480224609375, 0.276611328125, 0.75634765625, 0.99462890625],
 [22415, 13, 576, 6722, 315])

## Load large model (Qwen3-8B)

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM


main_model_name = "Qwen/Qwen3-8B"

tokenizer_main = AutoTokenizer.from_pretrained(main_model_name)
model_main = AutoModelForCausalLM.from_pretrained(
    main_model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print(f"Loaded: {main_model_name}")

Loading weights: 100%|██████████| 399/399 [00:04<00:00, 94.78it/s] 


Loaded: Qwen/Qwen3-8B


## Test main model generation

In [44]:
prompt = "The capital of Russia is"

inputs = tokenizer_main(prompt, return_tensors="pt").to('cuda')

with torch.no_grad():
    outputs = model_main(**inputs)

logits = outputs.logits
last_logit = logits[0, -1, :]

probs = torch.softmax(last_logit, dim = -1)
best_token = torch.topk(probs, 1)

best_indx = best_token.indices[0].item()
best_prob = best_token.values[0].item()

best_indx, best_prob

(22415, 0.44921875)

In [45]:
tokenizer_main.decode(best_indx)

' Moscow'

## Compare both model on the same promt

In [51]:
prompt = "The capital of Russia is"

_, answer, prob = generate_recs_draft_model(draft_model, prompt, 1)

inputs = tokenizer_main(prompt, return_tensors="pt").to('cuda')

with torch.no_grad():
    outputs = model_main(**inputs)

logits = outputs.logits
last_logit = logits[0, -1, :]

probs = torch.softmax(last_logit, dim = -1)
best_token = torch.topk(probs, 1)

best_indx = best_token.indices[0].item()
best_prob = best_token.values[0].item()

print(f"Small model answer: {answer} | prob: {prob[0]}")
print(f"Big model answer: {tokenizer_main.decode(best_indx)} | prob: {best_prob}")
print(f"Ratio = {best_prob / prob[0]:.3f}")

Small model answer:  Moscow | prob: 0.38623046875
Big model answer:  Moscow | prob: 0.44921875
Ratio = 1.163


## Create func to generate token with big model

In [58]:
def generate_token_main_model(model, tokenizer, text_before: str = 'I want to'):

    inputs = tokenizer(text_before, return_tensors="pt").to('cuda')

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    last_logit = logits[0, -1, :]

    probs = torch.softmax(last_logit, dim = -1)
    best_token = torch.topk(probs, 1)

    best_indx = best_token.indices[0].item()
    best_prob = best_token.values[0].item()

    return tokenizer.decode(best_indx), best_indx, best_prob

In [59]:
generate_token_main_model(model_main, tokenizer_main)

(' create', 1855, 0.150634765625)

## Create func to verify all tokens

In [178]:
def verify_all_tokens(main_model, tokenizer, prompt, draft_ids):
    
    draft_text = tokenizer.decode(draft_ids)
    full_text = prompt + draft_text
    
    inputs = tokenizer(full_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = main_model(**inputs)
    
    probs = torch.softmax(outputs.logits[0], dim=-1)  # [seq_len, vocab_size]
    
    n_prompt = tokenizer(prompt, return_tensors="pt").input_ids.shape[1]
    
    main_probs = []
    for i, tok_id in enumerate(draft_ids):
        prob = probs[n_prompt - 1 + i, tok_id].item()
        main_probs.append(prob)
    
    return main_probs

## Create func to generate big model answer with draft model help

In [179]:
def speculative_decode(prompt, draft_model, main_model, tokenizer, n_tokens: int = 20, K: int = 5):

    for _ in range(n_tokens // K):

        _, draft_tokens, draft_probs, draft_tokens_ids = generate_recs_draft_model(draft_model, prompt, K)
        main_probs = verify_all_tokens(main_model, tokenizer_main, prompt, draft_tokens_ids)

        for i in range(K):

            ratio = main_probs[i] / draft_probs[i]
            token_str = tokenizer.decode(draft_tokens_ids[i])

            if ratio >= 0.8:
                prompt += token_str
                
            else:
                new_word, _, _ = generate_token_main_model(main_model, tokenizer, prompt)
                prompt += new_word
                break
            
    return prompt

In [180]:
speculative_decode("The capital of Russia is", draft_model, model_main, tokenizer, n_tokens=40, K=5)

'The capital of Russia is Moscow. The capital of the United States is Washington, D.C. The capital of France is Paris. The capital of Germany is Berlin.'

In [181]:
import time

prompt = "The capital of Russia is"
N = 1000

start = time.time()
text = prompt
for _ in range(N):
    token, _, _ = generate_token_main_model(model_main, tokenizer_main, text)
    text += token
    
normal_time = time.time() - start
print(f"Base: {normal_time:.2f}s. Text: '{text}'")

start = time.time()
result = speculative_decode(prompt, draft_model, model_main, tokenizer_main, n_tokens=N, K=5)
spec_time = time.time() - start
print(f"Speculative: {spec_time:.2f}s. Text: '{result}'")

print(f"Speedup: {normal_time / spec_time:.2f}x")

Base: 159.63s. Text: 'The capital of Russia is Moscow. The capital of the United States is Washington, D.C. The capital of France is Paris. The capital of Japan is Tokyo. The capital of Brazil is Brasília. The capital of Canada is Ottawa. The capital of Australia is Canberra. The capital of Germany is Berlin. The capital of Italy is Rome. The capital of Spain is Madrid. The capital of Mexico is Mexico City. The capital of India is New Delhi. The capital of China is Beijing. The capital of South Korea is Seoul. The capital of South Africa is Pretoria. The capital of Nigeria is Abuja. The capital of Egypt is Cairo. The capital of Saudi Arabia is Riyadh. The capital of Iran is Tehran. The capital of Iraq is Baghdad. The capital of Syria is Damascus. The capital of Turkey is Ankara. The capital of Greece is Athens. The capital of Poland is Warsaw. The capital of Hungary is Budapest. The capital of Czech Republic is Prague. The capital of Austria is Vienna. The capital of Switzerland is Ber